# Model 4 Training: GAN (Generative Adversarial Network) for Image Restoration (50 Epochs with Early Stopping)

This notebook trains an advanced **Conditional Generative Adversarial Network (cGAN)** for joint image restoration (Noise + Blur + Low-Light) on Kaggle dual GPUs (T4 x 2) for up to **50 Epochs** with **Automated Early Stopping**:

- **Generator**: Dense Residual-in-Residual Dense Block (RRDB) network with 8 dense blocks, residual scaling (0.2), and direct skip connection from head to tail.
- **Discriminator**: Markovian 70x70 PatchGAN discriminator operating on $14 \times 14$ receptive patches for sharp texture and edge fidelity.
- **Loss Function**: Multi-scale hybrid objective:
  - $\mathcal{L}_{\text{pixel}}$: L1 Reconstruction Loss (weight = 1.0)
  - $\mathcal{L}_{\text{perceptual}}$: VGG-19 Feature Perceptual Loss up to conv3_4 (weight = 0.1)
  - $\mathcal{L}_{\text{adv}}$: Least-Squares GAN (LSGAN / MSE) Adversarial Loss (weight = 0.001)
- **Automated Early Stopping**:
  - Metric: Validation PSNR (`val_psnr`)
  - Patience: 8 epochs without improvement ($\ge 0.05$ dB)
  - Automatically halts training when the generator reaches its optimal reconstruction fidelity, preventing discriminator overpowering and wasted compute.
- **Adaptive LR Scheduling**: `ReduceLROnPlateau` halves the learning rate when validation PSNR plateaus, allowing the generator to settle into finer local minima.
- **Best Model Checkpoint Restoration**: Ensures test evaluation and exported zip contain the peak-performing generator weights.
- **Dual-GPU Multi-Threading**: DataParallel across both T4 GPUs with memory cleanup and cell timing.


In [ ]:
import time
GLOBAL_START_TIME = time.time()
_cell_start = time.time()

import os
import gc
import glob
import json
import random
import zipfile
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as torchvision_models

# Set seeds for deterministic reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

def print_timer(cell_name, cell_start):
    cell_elapsed = time.time() - cell_start
    total_elapsed = time.time() - GLOBAL_START_TIME
    print(f"[Cell Timer - {cell_name}] Cell Time: {cell_elapsed:.2f}s | Cumulative Time: {total_elapsed:.2f}s")

print_timer("Imports and Setup", _cell_start)


In [ ]:
_cell_start = time.time()

# Environment and Dataset Path Auto-Discovery Resolution
def find_degraded_dataset_root():
    search_candidates = [
        "/kaggle/input/datasets/hetulkadiya/degraded-dataset-zip-file/degraded_dataset",
        "/kaggle/input/degraded-dataset-zip-file/degraded_dataset",
        "/kaggle/input/degraded_dataset",
        "../data/degraded_dataset",
        "./degraded_dataset",
        "backend/degraded_dataset",
        "../backend/degraded_dataset"
    ]
    for cand in search_candidates:
        if os.path.exists(cand):
            return cand

    for search_base in ["/kaggle/input", "..", "."]:
        if os.path.exists(search_base):
            for root, dirs, files in os.walk(search_base):
                if "clean" in dirs or "c1_only_noise" in dirs or ("train" in dirs and "val" in dirs):
                    return root

    return search_candidates[0]

DATASET_ROOT = find_degraded_dataset_root()
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# Multi-GPU Accelerator Setup
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

def get_compute_device():
    if NUM_GPUS > 0:
        try:
            for i in range(NUM_GPUS):
                gpu_name = torch.cuda.get_device_name(i)
                print(f"Detected GPU {i}: {gpu_name}")

            test_tensor = torch.randn(2, 3, 16, 16, device="cuda:0")
            test_conv = nn.Conv2d(3, 16, 3, padding=1).to("cuda:0")
            _ = test_conv(test_tensor)
            return torch.device("cuda")
        except Exception as e:
            print(f"Warning: CUDA error encountered on current GPU ({e}).")
            print("Recommendation: In Kaggle Notebook Settings (right sidebar), switch Accelerator to 'GPU T4 x 2'.")
            return torch.device("cpu")
    return torch.device("cpu")

DEVICE = get_compute_device()
print(f"Active Main Device: {DEVICE} (Total GPUs Available: {NUM_GPUS})")
print(f"Dataset Root: {DATASET_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

print_timer("Path and Multi-GPU Device Setup", _cell_start)


In [ ]:
_cell_start = time.time()

# Robust PyTorch Dataset for Paired Clean and 7-Combo Degraded Images
class KaggleDegradedDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(128, 128)):
        self.target_size = target_size
        self.pairs = []

        # Discover split directory
        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            candidates = [d for d in os.listdir(root_dir)
                          if d.lower() == split.lower() and os.path.isdir(os.path.join(root_dir, d))]
            split_dir = os.path.join(root_dir, candidates[0]) if candidates else root_dir

        # Discover clean ground truth directory
        clean_dir = None
        for clean_name in ["clean", "ground_truth", "gt", "clean_images", "original"]:
            test_path = os.path.join(split_dir, clean_name)
            if os.path.exists(test_path):
                clean_dir = test_path
                break

        if clean_dir is None:
            for r, dirs, files in os.walk(split_dir):
                for d in dirs:
                    if d.lower() in ["clean", "ground_truth", "gt"]:
                        clean_dir = os.path.join(r, d)
                        break
                if clean_dir:
                    break

        combos = [
            "c1_only_noise", "c2_only_blur", "c3_only_low_light",
            "c4_noise_blur", "c5_noise_low_light", "c6_blur_low_light",
            "c7_all_three"
        ]

        if clean_dir and os.path.exists(clean_dir):
            clean_files = sorted(
                glob.glob(os.path.join(clean_dir, "*.[pP][nN][gG]")) +
                glob.glob(os.path.join(clean_dir, "*.[jJ][pP][gG]")) +
                glob.glob(os.path.join(clean_dir, "*.[jJ][pP][eE][gG]"))
            )
            parent_dir = os.path.dirname(clean_dir)
            subfolders = [d for d in os.listdir(parent_dir)
                          if os.path.isdir(os.path.join(parent_dir, d)) and d != os.path.basename(clean_dir)]

            for clean_path in clean_files:
                filename = os.path.basename(clean_path)
                for combo in combos:
                    deg_path = os.path.join(parent_dir, combo, filename)
                    if os.path.exists(deg_path):
                        self.pairs.append((deg_path, clean_path, combo))
                if len(self.pairs) == 0:
                    for sub in subfolders:
                        deg_path = os.path.join(parent_dir, sub, filename)
                        if os.path.exists(deg_path):
                            self.pairs.append((deg_path, clean_path, sub))

        self.to_tensor = T.Compose([T.Resize(target_size), T.ToTensor()])
        print(f"Loaded [{split.upper()}] dataset split from '{split_dir}': {len(self.pairs)} paired samples.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, clean_path, combo = self.pairs[idx]
        with Image.open(deg_path) as deg_img, Image.open(clean_path) as clean_img:
            deg_tensor = self.to_tensor(deg_img.convert("RGB"))
            clean_tensor = self.to_tensor(clean_img.convert("RGB"))
        return deg_tensor, clean_tensor

# Create Datasets and DataLoaders
USE_CUDA = (DEVICE.type == "cuda")
BATCH_SIZE = 32 if NUM_GPUS > 1 else (16 if USE_CUDA else 4)
NUM_WORKERS = (2 * NUM_GPUS) if USE_CUDA else 0

train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
val_dataset   = KaggleDegradedDataset(DATASET_ROOT, split="val")
test_dataset  = KaggleDegradedDataset(DATASET_ROOT, split="test")

# Fallback deep dynamic path search if 0 pairs found
if len(train_dataset) == 0:
    print("Warning: Initial dataset discovery returned 0 samples. Executing deep dynamic path search...")
    for root, dirs, files in os.walk(DATASET_ROOT):
        if "clean" in dirs:
            DATASET_ROOT = os.path.dirname(root) if os.path.basename(root) in ["train", "val", "test"] else root
            print(f"Updated DATASET_ROOT to: {DATASET_ROOT}")
            train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
            val_dataset   = KaggleDegradedDataset(DATASET_ROOT, split="val")
            test_dataset  = KaggleDegradedDataset(DATASET_ROOT, split="test")
            break

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": USE_CUDA,
    "persistent_workers": (NUM_WORKERS > 0)
}

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True,  **loader_kwargs)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kwargs)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **loader_kwargs)

print(f"DataLoader Configured: Batch Size = {BATCH_SIZE} | Workers = {NUM_WORKERS} | Persistent = {NUM_WORKERS > 0}")
print(f"Dataset Pair Counts -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print_timer("Dataset and DataLoaders Setup", _cell_start)


In [ ]:
_cell_start = time.time()

# Model 4: GAN Architecture - Dense RRDB Generator and PatchGAN Discriminator

class DenseBlock(nn.Module):
    """Residual Dense Block with dense connections and residual scaling."""
    def __init__(self, channels=64, growth_rate=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, growth_rate, 3, padding=1)
        self.conv2 = nn.Conv2d(channels + growth_rate, growth_rate, 3, padding=1)
        self.conv3 = nn.Conv2d(channels + 2 * growth_rate, channels, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], dim=1)))
        x3 = self.conv3(torch.cat([x, x1, x2], dim=1))
        return x3 * 0.2 + x

class RestorationGANGenerator(nn.Module):
    """
    Model 4 Generator: Dense Residual Block-based GAN Generator.
    Input: Degraded Image [B, 3, H, W] -> Output: Restored Image [B, 3, H, W]
    """
    def __init__(self, in_channels=3, out_channels=3, num_blocks=8, base_channels=64):
        super(RestorationGANGenerator, self).__init__()
        self.head = nn.Conv2d(in_channels, base_channels, 3, padding=1)
        self.body = nn.Sequential(*[DenseBlock(base_channels) for _ in range(num_blocks)])
        self.conv_body = nn.Conv2d(base_channels, base_channels, 3, padding=1)
        self.tail = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, out_channels, 3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        feat = self.head(x)
        res = self.conv_body(self.body(feat))
        out = self.tail(feat + res)
        return out

class PatchDiscriminator(nn.Module):
    """
    Markovian PatchGAN Discriminator for fine visual detail discrimination.
    Outputs 14x14 local validity map for 128x128 input images.
    """
    def __init__(self, in_channels=3, base_channels=64):
        super(PatchDiscriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, base_channels, 4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels, base_channels * 2, 4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels * 2, base_channels * 4, 4, stride=2, padding=1),
            nn.BatchNorm2d(base_channels * 4),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(base_channels * 4, 1, 4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.net(x)

class VGGPerceptualLoss(nn.Module):
    """Perceptual VGG-19 Feature Loss (features up to layer 16)."""
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        try:
            vgg = torchvision_models.vgg19(weights=torchvision_models.VGG19_Weights.DEFAULT)
        except Exception:
            vgg = torchvision_models.vgg19(pretrained=True)
        self.feature_extractor = nn.Sequential(*list(vgg.features[:16])).eval()
        for param in self.feature_extractor.parameters():
            param.requires_grad = False

    def forward(self, restored, clean):
        mean = torch.tensor([0.485, 0.456, 0.406], device=restored.device).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=restored.device).view(1, 3, 1, 1)
        restored_feat = self.feature_extractor((restored - mean) / std)
        clean_feat    = self.feature_extractor((clean - mean) / std)
        return F.l1_loss(restored_feat, clean_feat)

# Initialize Generator and Discriminator
base_generator     = RestorationGANGenerator(num_blocks=8)
base_discriminator = PatchDiscriminator()

# Wrap both with DataParallel if multiple GPUs are detected
if NUM_GPUS > 1:
    print(f"Enabling Multi-GPU DataParallel across {NUM_GPUS} GPUs for Generator and Discriminator!")
    generator     = nn.DataParallel(base_generator)
    discriminator = nn.DataParallel(base_discriminator)
else:
    generator     = base_generator
    discriminator = base_discriminator

generator     = generator.to(DEVICE)
discriminator = discriminator.to(DEVICE)

gen_params  = sum(p.numel() for p in base_generator.parameters() if p.requires_grad)
disc_params = sum(p.numel() for p in base_discriminator.parameters() if p.requires_grad)
print(f"Generator Trainable Parameters    : {gen_params:,}")
print(f"Discriminator Trainable Parameters: {disc_params:,}")

print_timer("Multi-GPU GAN Model Setup", _cell_start)


In [ ]:
_cell_start = time.time()

# Loss Functions, Optimizers, Learning Rate Schedulers, and Metric Setup

# VGG Perceptual Loss (move feature extractor to correct device)
perceptual_loss_fn  = VGGPerceptualLoss().to(DEVICE)
pixel_loss_fn       = nn.L1Loss()
adversarial_loss_fn = nn.MSELoss()  # LSGAN Adversarial Loss

# Loss weights balancing pixel accuracy, structural texture, and perceptual realism
LAMBDA_PIXEL       = 1.0
LAMBDA_PERCEPTUAL  = 0.1
LAMBDA_ADVERSARIAL = 0.001

# Separate optimizers for Generator and Discriminator
optimizer_G = torch.optim.Adam(generator.parameters(),     lr=1e-4, betas=(0.9, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.9, 0.999))

# Learning Rate Schedulers (halve LR if validation PSNR plateaus for 4 epochs)
scheduler_G = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_G, mode="max", factor=0.5, patience=4, min_lr=1e-6
)
scheduler_D = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_D, mode="max", factor=0.5, patience=4, min_lr=1e-6
)

def calculate_batch_psnr(pred, target):
    with torch.no_grad():
        mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3])
        psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
        return torch.mean(psnr).item()

def calculate_batch_ssim(pred, target):
    with torch.no_grad():
        mu_x  = F.avg_pool2d(pred, 3, 1, 1)
        mu_y  = F.avg_pool2d(target, 3, 1, 1)
        sigma_x  = F.avg_pool2d(pred * pred, 3, 1, 1) - mu_x ** 2
        sigma_y  = F.avg_pool2d(target * target, 3, 1, 1) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target, 3, 1, 1) - mu_x * mu_y
        C1, C2  = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / \
                   ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
        return torch.mean(ssim_map).item()

def extract_clean_state_dict(m):
    return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()

def load_clean_state_dict(m, path, device):
    state_dict = torch.load(path, map_location=device)
    if isinstance(m, nn.DataParallel):
        m.module.load_state_dict(state_dict)
    else:
        m.load_state_dict(state_dict)

print(f"Loss Weights -> Pixel (L1): {LAMBDA_PIXEL} | Perceptual (VGG): {LAMBDA_PERCEPTUAL} | Adversarial (LSGAN): {LAMBDA_ADVERSARIAL}")
print("Adaptive Learning Rate Schedulers configured with ReduceLROnPlateau (factor=0.5, patience=4)")
print_timer("Loss Functions and Optimizers Setup", _cell_start)


In [ ]:
_cell_start = time.time()

# 50-Epoch Multi-GPU GAN Training and Validation Loop with Automated Early Stopping
EPOCHS = 50
PATIENCE = 8            # Stop if validation PSNR does not improve for 8 consecutive epochs
MIN_DELTA = 0.05        # Minimum PSNR improvement in dB to count as a significant gain

best_val_psnr    = 0.0
best_val_ssim    = 0.0
best_epoch       = 0
patience_counter = 0
early_stopped    = False

GEN_SAVE_PATH  = os.path.join(OUTPUT_DIR, "gan_generator.pth")
DISC_SAVE_PATH = os.path.join(OUTPUT_DIR, "gan_discriminator.pth")

history = {
    "train_g_loss": [],
    "train_d_loss": [],
    "val_psnr": [],
    "val_ssim": [],
    "lr_g": [],
    "lr_d": []
}

print(f"Starting Multi-GPU GAN Training for up to {EPOCHS} Epochs on {max(NUM_GPUS, 1)} device(s)...")
print(f"Early Stopping Configuration -> Metric: val_psnr | Patience: {PATIENCE} epochs | Min Delta: {MIN_DELTA} dB")
print("=" * 85)

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # ------------------ Training Phase ------------------
    generator.train()
    discriminator.train()

    total_g_loss = 0.0
    total_d_loss = 0.0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Train]", leave=False, mininterval=1.0)

    for degraded, clean in train_pbar:
        degraded = degraded.to(DEVICE, non_blocking=True)
        clean    = clean.to(DEVICE, non_blocking=True)

        # --- Step 1: Train Discriminator ---
        optimizer_D.zero_grad(set_to_none=True)
        with torch.no_grad():
            restored_fake = generator(degraded)

        real_pred = discriminator(clean)
        fake_pred = discriminator(restored_fake.detach())

        d_loss_real = adversarial_loss_fn(real_pred, torch.ones_like(real_pred))
        d_loss_fake = adversarial_loss_fn(fake_pred, torch.zeros_like(fake_pred))
        d_loss = (d_loss_real + d_loss_fake) * 0.5

        d_loss.backward()
        torch.nn.utils.clip_grad_norm_(discriminator.parameters(), max_norm=1.0)
        optimizer_D.step()

        # --- Step 2: Train Generator ---
        optimizer_G.zero_grad(set_to_none=True)
        restored = generator(degraded)

        fake_pred_for_gen = discriminator(restored)
        g_loss_adv        = adversarial_loss_fn(fake_pred_for_gen, torch.ones_like(fake_pred_for_gen))
        g_loss_pixel      = pixel_loss_fn(restored, clean)
        g_loss_perceptual = perceptual_loss_fn(restored, clean)

        g_loss = (LAMBDA_PIXEL * g_loss_pixel
                  + LAMBDA_PERCEPTUAL * g_loss_perceptual
                  + LAMBDA_ADVERSARIAL * g_loss_adv)

        g_loss.backward()
        torch.nn.utils.clip_grad_norm_(generator.parameters(), max_norm=1.0)
        optimizer_G.step()

        total_g_loss += g_loss.item()
        total_d_loss += d_loss.item()

        train_pbar.set_postfix({"G_Loss": f"{g_loss.item():.4f}", "D_Loss": f"{d_loss.item():.4f}"})

    avg_g_loss = total_g_loss / max(len(train_loader), 1)
    avg_d_loss = total_d_loss / max(len(train_loader), 1)
    curr_lr_g  = optimizer_G.param_groups[0]["lr"]
    curr_lr_d  = optimizer_D.param_groups[0]["lr"]

    history["train_g_loss"].append(round(avg_g_loss, 4))
    history["train_d_loss"].append(round(avg_d_loss, 4))
    history["lr_g"].append(round(curr_lr_g, 8))
    history["lr_d"].append(round(curr_lr_d, 8))

    # ------------------ Validation Phase ------------------
    generator.eval()
    val_psnr = 0.0
    val_ssim = 0.0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d} [Val]  ", leave=False, mininterval=1.0)

    with torch.no_grad():
        for degraded, clean in val_pbar:
            degraded = degraded.to(DEVICE, non_blocking=True)
            clean    = clean.to(DEVICE, non_blocking=True)
            restored = generator(degraded)
            val_psnr += calculate_batch_psnr(restored, clean)
            val_ssim += calculate_batch_ssim(restored, clean)

    val_psnr /= max(len(val_loader), 1)
    val_ssim /= max(len(val_loader), 1)
    history["val_psnr"].append(round(val_psnr, 2))
    history["val_ssim"].append(round(val_ssim, 4))

    # Step adaptive learning rate schedulers
    scheduler_G.step(val_psnr)
    scheduler_D.step(val_psnr)

    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_time:.1f}s] -> G Loss: {avg_g_loss:.4f} | D Loss: {avg_d_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} | LR: {curr_lr_g:.2e}")

    # ------------------ Best Checkpoint & Early Stopping ------------------
    if val_psnr > (best_val_psnr + MIN_DELTA):
        best_val_psnr    = val_psnr
        best_val_ssim    = val_ssim
        best_epoch       = epoch
        patience_counter = 0
        torch.save(extract_clean_state_dict(generator),     GEN_SAVE_PATH)
        torch.save(extract_clean_state_dict(discriminator), DISC_SAVE_PATH)
        print(f"  --> [BEST CHECKPOINT] Saved model weights at Epoch {epoch:02d} (Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f})")
    else:
        patience_counter += 1
        if val_psnr > best_val_psnr:
            # Marginally better (< MIN_DELTA), save weights but do not reset patience counter
            best_val_psnr = val_psnr
            best_val_ssim = val_ssim
            best_epoch    = epoch
            torch.save(extract_clean_state_dict(generator),     GEN_SAVE_PATH)
            torch.save(extract_clean_state_dict(discriminator), DISC_SAVE_PATH)
            print(f"  --> [MARGINAL BEST] Updated weights (Val PSNR: {val_psnr:.2f} dB, +{val_psnr - best_val_psnr:.3f} dB)")
        print(f"  --> [Patience] No significant gain for {patience_counter}/{PATIENCE} epoch(s). Best is {best_val_psnr:.2f} dB at Epoch {best_epoch:02d}.")

        if patience_counter >= PATIENCE:
            print(f"\n[EARLY STOPPING TRIGGERED] Validation PSNR has not improved by >= {MIN_DELTA} dB for {PATIENCE} consecutive epochs.")
            print(f"Stopping training at Epoch {epoch:02d}. Best model weights were recorded at Epoch {best_epoch:02d} with Val PSNR {best_val_psnr:.2f} dB.")
            early_stopped = True
            break

    # Free unused VRAM between epochs
    if USE_CUDA:
        torch.cuda.empty_cache()
    gc.collect()

print("=" * 85)
if not early_stopped:
    print(f"Completed full scheduled run of {EPOCHS} epochs!")
print(f"Peak Validation Performance: Epoch {best_epoch:02d} -> Val PSNR: {best_val_psnr:.2f} dB | Val SSIM: {best_val_ssim:.4f}")
print_timer("50-Epoch GAN Training Loop with Early Stopping", _cell_start)


In [ ]:
_cell_start = time.time()

# Final Evaluation and Benchmark on Test Set (Using Peak Checkpoint)
if os.path.exists(GEN_SAVE_PATH):
    print(f"Loading peak checkpoint from '{GEN_SAVE_PATH}' (recorded at Epoch {best_epoch:02d})...")
    load_clean_state_dict(generator, GEN_SAVE_PATH, DEVICE)
else:
    print("Warning: Checkpoint file not found, evaluating with current model state.")

generator.eval()
deg_psnr_total, deg_ssim_total = 0.0, 0.0
res_psnr_total, res_ssim_total = 0.0, 0.0

with torch.no_grad():
    for degraded, clean in tqdm(test_loader, desc="Evaluating Test Set", mininterval=1.0):
        degraded = degraded.to(DEVICE, non_blocking=True)
        clean    = clean.to(DEVICE, non_blocking=True)
        restored = generator(degraded)

        deg_psnr_total += calculate_batch_psnr(degraded, clean)
        deg_ssim_total += calculate_batch_ssim(degraded, clean)

        res_psnr_total += calculate_batch_psnr(restored, clean)
        res_ssim_total += calculate_batch_ssim(restored, clean)

n_test_batches = max(len(test_loader), 1)
avg_deg_psnr   = deg_psnr_total / n_test_batches
avg_deg_ssim   = deg_ssim_total / n_test_batches
avg_res_psnr   = res_psnr_total / n_test_batches
avg_res_ssim   = res_ssim_total / n_test_batches
psnr_gain      = avg_res_psnr - avg_deg_psnr
ssim_gain      = avg_res_ssim - avg_deg_ssim

print("\n======================================================")
print("MODEL 4 (RESTORATION GAN) 50-EPOCH TEST BENCHMARK")
print("======================================================")
print(f"GPUs Utilized           : {max(NUM_GPUS, 1)}")
print(f"Configured Epochs       : {EPOCHS} (Early Stopping Patience = {PATIENCE})")
print(f"Actual Epochs Trained   : {len(history['val_psnr'])}")
print(f"Early Stopped           : {early_stopped} (Best Epoch: {best_epoch})")
print(f"Degraded Input Quality  : PSNR = {avg_deg_psnr:.2f} dB | SSIM = {avg_deg_ssim:.4f}")
print(f"GAN Restored Quality    : PSNR = {avg_res_psnr:.2f} dB | SSIM = {avg_res_ssim:.4f}")
print(f"Restoration dB Gain     : PSNR Gain = +{psnr_gain:.2f} dB | SSIM Gain = +{ssim_gain:.4f}")
print("======================================================")

# Save Comprehensive Evaluation Metrics JSON
metrics_json_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
metrics_data = {
    "model_name": "Restoration GAN (Dense RRDB Generator + PatchGAN Discriminator)",
    "gpus_used": max(NUM_GPUS, 1),
    "epochs_configured": EPOCHS,
    "epochs_trained": len(history["val_psnr"]),
    "early_stopping_triggered": early_stopped,
    "best_epoch": best_epoch,
    "best_val_psnr": best_val_psnr,
    "best_val_ssim": best_val_ssim,
    "early_stopping_params": {
        "patience": PATIENCE,
        "min_delta": MIN_DELTA
    },
    "batch_size": BATCH_SIZE,
    "loss_weights": {
        "pixel_l1": LAMBDA_PIXEL,
        "perceptual_vgg": LAMBDA_PERCEPTUAL,
        "adversarial_lsgan": LAMBDA_ADVERSARIAL
    },
    "optimizer": "Adam (lr=1e-4) with ReduceLROnPlateau",
    "history": history,
    "test_results": {
        "degraded_psnr": round(avg_deg_psnr, 2),
        "degraded_ssim": round(avg_deg_ssim, 4),
        "restored_psnr": round(avg_res_psnr, 2),
        "restored_ssim": round(avg_res_ssim, 4),
        "psnr_gain_db": round(psnr_gain, 2),
        "ssim_gain": round(ssim_gain, 4)
    }
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

print(f"Metrics successfully exported to: {metrics_json_path}")
print_timer("Test Evaluation and Metrics Export", _cell_start)


In [ ]:
_cell_start = time.time()

# Plot Training Curves and Generate Sample Visual Restorations
num_epochs_done = len(history["val_psnr"])
epochs_range    = range(1, num_epochs_done + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Generator and Discriminator Loss Curves
axes[0].plot(epochs_range, history["train_g_loss"], label="Generator Loss",     marker="o", markersize=3, color="#ef4444")
axes[0].plot(epochs_range, history["train_d_loss"], label="Discriminator Loss", marker="s", markersize=3, color="#3b82f6")
axes[0].set_title("GAN Generator and Discriminator Loss", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# PSNR Curve with Best Checkpoint Marker
axes[1].plot(epochs_range, history["val_psnr"], label="Val PSNR (dB)", marker="^", markersize=3, color="#10b981")
if best_epoch > 0 and best_epoch <= num_epochs_done:
    axes[1].scatter([best_epoch], [best_val_psnr], color="gold", edgecolor="black", s=140, zorder=5, label=f"Best (Ep {best_epoch}: {best_val_psnr:.2f}dB)")
axes[1].set_title("Validation PSNR (dB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

# SSIM Curve
axes[2].plot(epochs_range, history["val_ssim"], label="Val SSIM", marker="d", markersize=3, color="#8b5cf6")
axes[2].set_title("Validation SSIM Score", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM Score")
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plots_image_path = os.path.join(OUTPUT_DIR, "loss_psnr_ssim_curves.png")
plt.savefig(plots_image_path, dpi=300)
plt.show()

# Visual Restoration Samples Plot
generator.eval()
deg_b, clean_b = None, None
for deg_sample, clean_sample in test_loader:
    deg_b = deg_sample.to(DEVICE)
    clean_b = clean_sample.to(DEVICE)
    with torch.no_grad():
        res_b = generator(deg_b)
    break

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, deg_b.shape[0])):
    deg_np   = deg_b[i].cpu().permute(1, 2, 0).numpy()
    res_np   = res_b[i].cpu().permute(1, 2, 0).numpy()
    clean_np = clean_b[i].cpu().permute(1, 2, 0).numpy()

    axes[i, 0].imshow(np.clip(clean_np, 0.0, 1.0))
    axes[i, 0].set_title("Original Clean", fontsize=9)
    axes[i, 0].axis("off")

    axes[i, 1].imshow(np.clip(deg_np, 0.0, 1.0))
    axes[i, 1].set_title("Degraded Input", fontsize=9)
    axes[i, 1].axis("off")

    axes[i, 2].imshow(np.clip(res_np, 0.0, 1.0))
    axes[i, 2].set_title("GAN Restored", fontsize=9)
    axes[i, 2].axis("off")

plt.suptitle("Restoration GAN 50-Epoch Test Sample Restorations", fontsize=13)
plt.tight_layout()
samples_image_path = os.path.join(OUTPUT_DIR, "test_restoration_samples.png")
plt.savefig(samples_image_path, dpi=300)
plt.show()

print_timer("Plots and Visual Restorations", _cell_start)


In [ ]:
_cell_start = time.time()

# Package Checkpoints, Metrics, Plots into gan_restoration_results.zip
ZIP_FILE_PATH = os.path.join(OUTPUT_DIR, "gan_restoration_results.zip")

files_to_zip = [
    (GEN_SAVE_PATH,        "gan_generator.pth"),
    (DISC_SAVE_PATH,       "gan_discriminator.pth"),
    (metrics_json_path,    "training_metrics.json"),
    (plots_image_path,     "loss_psnr_ssim_curves.png"),
    (samples_image_path,   "test_restoration_samples.png")
]

with zipfile.ZipFile(ZIP_FILE_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for src_path, arcname in files_to_zip:
        if os.path.exists(src_path):
            zipf.write(src_path, arcname=arcname)
            print(f"Added to Zip: {arcname}")

# Also create a copy named gan_restoration_results_50epochs.zip for explicit versioning
ZIP_50_PATH = os.path.join(OUTPUT_DIR, "gan_restoration_results_50epochs.zip")
try:
    import shutil
    shutil.copyfile(ZIP_FILE_PATH, ZIP_50_PATH)
except Exception:
    pass

print(f"\n======================================================")
print(f"RESULTS ZIP ARCHIVE CREATED SUCCESSFULLY")
print(f"Zip Path: {ZIP_FILE_PATH}")
print(f"Size: {os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024):.2f} MB")
print(f"Contains: gan_generator.pth, gan_discriminator.pth, training_metrics.json, curves, visual samples")
print(f"======================================================")

print_timer("Zip Packaging and Export", _cell_start)
